# Credit Card Fraud Detection with XGBoost

This notebook demonstrates how to process the European cardholder credit
card transaction dataset and train a classical XGBoost classifier for
fraud detection. We perform a time-aware split to avoid data leakage,
scale the features, tune the model using randomized search, and
evaluate it on a held-out test set using metrics tailored for highly
imbalanced datasets (AUROC, AUPRC, recall, precision, F1, etc.).

**Note:** To run this notebook, download the `creditcard.csv` dataset
from the Kaggle competition page (European cardholders credit card
fraud detection) and place it in the `../data/raw/` directory relative
to this notebook.


In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from xgboost import XGBClassifier

# Load the dataset
# The dataset should be placed in the data/raw directory relative to this notebook
file_path = '../data/raw/creditcard.csv'

# Read the data
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    raise FileNotFoundError("Dataset not found. Please place creditcard.csv in ../data/raw/")

# Ensure the 'Time' column exists for the time-aware split
if 'Time' not in df.columns:
    raise KeyError("The dataset must contain a 'Time' column for time-aware splitting.")

# Sort by time to prevent future data leaking into the past
df_sorted = df.sort_values('Time').reset_index(drop=True)

# Split into train and test (80/20)
n_samples = len(df_sorted)
split_index = int(0.8 * n_samples)
train_df = df_sorted.iloc[:split_index].copy()
test_df = df_sorted.iloc[split_index:].copy()

# Separate features and labels
feature_cols = [col for col in train_df.columns if col != 'Class']
X_train = train_df[feature_cols].values
y_train = train_df['Class'].values
X_test = test_df[feature_cols].values
y_test = test_df['Class'].values

# Standardize features (important for tree-based methods and comparability)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with scale_pos_weight
# scale_pos_weight = (negative classes / positive classes)
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

# Define the XGBoost model
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    random_state=42,
)

# Hyperparameter search space (tuned for demonstration, expand as needed)
param_dist = {
    'n_estimators': [200, 400, 600],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'gamma': [0, 1, 5]
}

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring='average_precision',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1,
)

# Fit the model with hyperparameter tuning
search.fit(X_train_scaled, y_train)

# Best estimator after tuning
best_model = search.best_estimator_
print("Best parameters:", search.best_params_)

# Predict probabilities for the positive class on the test set
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

# Compute evaluation metrics
metrics = {}
metrics['auroc'] = roc_auc_score(y_test, y_proba)
metrics['auprc'] = average_precision_score(y_test, y_proba)
metrics['accuracy'] = accuracy_score(y_test, y_pred)

# Precision/Recall/F1 metrics (macro and weighted)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

metrics['precision_macro'] = prec_macro
metrics['recall_macro'] = rec_macro
metrics['f1_macro'] = f1_macro
metrics['precision_weighted'] = prec_weighted
metrics['recall_weighted'] = rec_weighted
metrics['f1_weighted'] = f1_weighted

# Positive class (fraud) metrics
prec_pos, rec_pos, f1_pos, support_pos = precision_recall_fscore_support(y_test, y_pred, labels=[1], average=None, zero_division=0)
metrics['precision_fraud'] = prec_pos[0]
metrics['recall_fraud'] = rec_pos[0]
metrics['f1_fraud'] = f1_pos[0]
metrics['support_fraud'] = support_pos[0]

# Confusion matrix and classification report
cm = confusion_matrix(y_test, y_pred)
metrics['confusion_matrix'] = cm.tolist()
metrics['classification_report'] = classification_report(y_test, y_pred, digits=4, zero_division=0)

print("Evaluation metrics:")
for k, v in metrics.items():
    if k not in ['confusion_matrix', 'classification_report']:
        print(f"{k}: {v}")
print("Confusion Matrix:", metrics['confusion_matrix'])
print("Classification Report:", metrics['classification_report'])


Fitting 3 folds for each of 20 candidates, totalling 60 fits



The metrics above summarize the performance of the XGBoost classifier trained on the European cardholder credit card fraud dataset. 
You can expand the hyperparameter search and adjust the threshold to potentially improve recall or precision.

This notebook highlights the full end-to-end process: data loading, preprocessing (time-aware split, scaling), hyperparameter tuning, 
and evaluation with metrics tailored for imbalanced classification.
